# 02 – RQ1: pose estimation from a drone view
In this notebook, we measure how much a pose estimator pretrained on
ground-level images degrades on aerial images, and how much of that loss we
can recover.

- Part A: we compute the reference PCK of RTMPose on ground-level images
  from the COCO validation set.
- Part B: we run the same evaluation on aerial frames from the UAV-Human
  dataset.
- Part C: we label the Okutama video with the RTMPose model as a teacher,
  fine-tune a yolo11n-pose model on these pseudo-labels as a student, and
  evaluate it on held-out frames.


In [ ]:
# Install the packages we need.
!pip -q install ultralytics rtmlib onnxruntime-gpu
import torch
print('CUDA available:', torch.cuda.is_available())

# Load the project code.
from pathlib import Path
SRC_ZIP = None
if SRC_ZIP is None:
    from google.colab import files
    up = files.upload()
    SRC_ZIP = next(iter(up))
!mkdir -p /content/project && unzip -q -o "$SRC_ZIP" -d /content/project
import sys
sys.path.insert(0, '/content/project/src')
sys.path.insert(0, '/content/project')

# Results are saved to Google Drive so they survive a disconnect.
from google.colab import drive
drive.mount('/content/drive')
OUT = Path('/content/drive/MyDrive/sar_project_results'); OUT.mkdir(parents=True, exist_ok=True)


In [ ]:
# Part A: compute the ground-level reference PCK, which also checks the evaluator.
import config
from pathlib import Path
config.RAW_DIR = Path('/content/data/raw'); config.RAW_DIR.mkdir(parents=True, exist_ok=True)
config.TABLES_DIR = OUT
!curl -sL -o /content/coco_ann.zip http://images.cocodataset.org/annotations/annotations_trainval2017.zip
!cd /content/data/raw && unzip -q -o /content/coco_ann.zip annotations/person_keypoints_val2017.json
import eval_pose
eval_pose.RAW_DIR = config.RAW_DIR; eval_pose.TABLES_DIR = OUT
ref = eval_pose.validate_on_coco(max_persons=150, device='cuda')


In [ ]:
# Download Okutama-Action files from the public Dropbox folder.
OKUTAMA_BASE = ('https://www.dropbox.com/scl/fo/9qvpsb3fsamvqzsa12149/'
                'APTyV-f01XLnJ0WFpZSBLOE?preview={name}&rlkey=7u7131amaul29amyr4jbnnu03&dl=1')

def fetch_okutama(name, dest='/content/data/okutama'):
    """Download and unpack one Okutama archive unless it is already present."""
    import subprocess, pathlib
    d = pathlib.Path(dest); d.mkdir(parents=True, exist_ok=True)
    zp = d / name
    if not zp.exists():
        subprocess.run(['curl', '-L', '-o', str(zp), OKUTAMA_BASE.format(name=name)], check=True)
    subprocess.run(['unzip', '-q', '-o', str(zp), '-d', str(d)], check=True)
    return d


In [ ]:
# Part B: download the UAV-Human pose subset from the public Google Drive.
!pip -q install gdown
!gdown 1kWStmFjrN1Njf6mj4rTso6XPMULcFKS5 -O /content/PoseEstimation.zip
!mkdir -p /content/data/raw/uavhuman_pose && unzip -q -o /content/PoseEstimation.zip -d /content/data/raw/uavhuman_pose


In [ ]:
# Compute the zero-shot aerial PCK on all 22,476 frames.
# The downscale parameter shrinks people before pose estimation, which
# separates the scale part of the domain gap from the viewpoint part.
import numpy as np, cv2
from data.uavhuman import iter_dataset
from pose import PoseEstimator
from eval_pose import pck

def aerial_pck(limit=None, downscale=1.0, device='cuda'):
    """Run the pose estimator over the UAV-Human frames and return the PCK."""
    est = PoseEstimator(device=device)
    preds, gts, viss, boxes = [], [], [], []
    for img_path, persons in iter_dataset('/content/data/raw/uavhuman_pose', limit=limit):
        img = cv2.imread(str(img_path))
        if img is None: continue
        if downscale != 1.0:
            img = cv2.resize(img, None, fx=downscale, fy=downscale)
        for p in persons:
            box = p['box'] * downscale
            kp, _ = est(img, box[None])
            preds.append(kp[0]); gts.append(p['kpts'] * downscale)
            viss.append(p['vis']); boxes.append(box)
    return pck(np.stack(preds), np.stack(gts), np.stack(viss), np.stack(boxes))

aerial = aerial_pck(limit=None)             # The full zero-shot aerial PCK.
aerial_small = aerial_pck(limit=3000, downscale=0.35)  # The small-person regime.
print('AERIAL zero-shot:', aerial)
print('AERIAL @0.35 scale:', aerial_small)  # Compare both against Part A.


In [ ]:
# Part C: pseudo-label Okutama with the teacher model.
import numpy as np, cv2
fetch_okutama('Sample.zip')          # Use TrainSetVideos.zip for the full run.
from data.okutama import parse_annotations
from pose import PoseEstimator

video = '/content/data/okutama/1.1.1.mov'
labels = parse_annotations('/content/data/okutama/1.1.1.txt')
est = PoseEstimator(device='cuda')   # The teacher model.

# Build a YOLO-pose dataset from the ground-truth boxes and teacher keypoints.
from pathlib import Path
ds = Path('/content/data/okutama_pose'); (ds/'images/train').mkdir(parents=True, exist_ok=True)
(ds/'labels/train').mkdir(parents=True, exist_ok=True)
(ds/'images/val').mkdir(parents=True, exist_ok=True); (ds/'labels/val').mkdir(parents=True, exist_ok=True)
cap = cv2.VideoCapture(video); idx = 0; n = 0
while True:
    ok, img = cap.read()
    if not ok: break
    if idx in labels and idx % 5 == 0:
        H, W = img.shape[:2]
        boxes = np.array([b.box for b in labels[idx]], np.float32)
        kpts, scores = est(img, boxes)
        split = 'val' if idx % 25 == 0 else 'train'
        # One label line per person: the box followed by the 17 keypoints.
        lines = []
        for (x1,y1,x2,y2), kp, sc in zip(boxes, kpts, scores):
            if sc.mean() < 0.35: continue   # Keep only the confident pseudo-labels.
            cx,cy,w,h = ((x1+x2)/2/W,(y1+y2)/2/H,(x2-x1)/W,(y2-y1)/H)
            ks = ' '.join(f'{x/W:.5f} {y/H:.5f} {2 if s>0.35 else 0}' for (x,y),s in zip(kp,sc))
            lines.append(f'0 {cx:.5f} {cy:.5f} {w:.5f} {h:.5f} ' + ks)
        if lines:
            cv2.imwrite(str(ds/f'images/{split}/{idx:06d}.jpg'), img)
            (ds/f'labels/{split}/{idx:06d}.txt').write_text('\n'.join(lines))
            n += 1
    idx += 1
cap.release(); print(n, 'pseudo-labeled frames')
(ds/'okutama_pose.yaml').write_text(f'path: {ds}\ntrain: images/train\nval: images/val\nkpt_shape: [17, 3]\nnames:\n  0: person\n')


In [ ]:
# Fine-tune the student model on the pseudo-labels.
from ultralytics import YOLO
student = YOLO('yolo11n-pose.pt')
student.train(data=str(ds/'okutama_pose.yaml'), epochs=20, imgsz=1280, batch=8,
              device=0, project='/content/runs', name='pose_ft', exist_ok=True)
!cp /content/runs/pose_ft/weights/best.pt {OUT}/yolo11n_pose_okutama.pt


In [ ]:
# Plot PCK stratified by person height. This is the main RQ1 figure.
import matplotlib.pyplot as plt
# Add the measured results here: ground level from Part A, and the aerial
# zero-shot and fine-tuned numbers from Parts B and C.
results = {'Ground level (COCO)': ref}
fig, ax = plt.subplots(figsize=(6,4))
bins = [k for k in ref if k.startswith('PCK_h')]
for name, m in results.items():
    ax.plot(bins, [m[b] for b in bins], marker='o', label=name)
ax.set_xlabel('Person pixel height'); ax.set_ylabel(f"PCK@{ref['alpha']}")
ax.set_title('RQ1: pose accuracy against person scale'); ax.legend(); ax.grid(alpha=.3)
fig.tight_layout(); fig.savefig(OUT / 'rq1_pck_vs_scale.png', dpi=150)
